# Ephraim’s fraud models · PBI 08
**Kochuveettil Toji Ephraim** · AMLOPS CA1 · ML Experiments

Predict order fraud using order attributes and customer purchase history. Use the AIML CA1 Part A dummy classifier, seeded decision tree, classification report and confusion matrix. Replace the original random split and full-data imputation with the team’s chronological split and training-only preprocessing.

Han Qian’s PBI 06 supplies the callable interface pattern. Nithish’s PBI 07 motivates using the same validation period and keeping later dates within training for model decisions. His reported scores use a different fit window, so compare on an aligned fit window before attributing differences to the estimator.

Run from the AMLOPS repo root on the authorised base CSVs. Use the checkout containing PBI 02 and PBI 03 and its compatible pandas, NumPy, scikit-learn, matplotlib, seaborn and MLflow dependencies. Record your checkout, CSV hashes and package versions below. Set `MLFLOW_TRACKING_URI` to the confirmed shared endpoint outside the notebook before the evidence run.

[PBI 08](https://github.com/hqasdf/agilemlops-ca1/issues/8) · [Team README](https://github.com/hqasdf/agilemlops-ca1/blob/HEAD/README.md) · [PBI 07](https://github.com/hqasdf/agilemlops-ca1/blob/768f89aaef0964ee97a678b9e53eac6f82712a56/reports/PBI07_model_comparison.md)

## 1. Define the training routine
This cell defines the training functions in the notebook. Run it before loading and training on the base data.

The routine uses PBI 03’s seven order-time features and adds days since first recorded purchase. Missing or future first-order dates are treated as unknown. IDs are used for joins. Snapshot aggregates and post-order information are excluded. The customer feature uses the same point-in-time definition documented by PBI 07.

In [ ]:
import os
from importlib.metadata import version
from urllib.parse import urlsplit

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier

from src.feature_engineering import (
    CUSTOMER_KEY, INPUT_FEATURES, ORDER_KEY, fit_preprocessor, prepare_training_data,
)

HISTORY = "days_since_first_order"
THRESHOLD = 0.50


def prepare_ephraim_data(customers, orders, labels):
    x_train, y_train, x_val, y_val, _, _, report, dates = prepare_training_data(
        customers, orders, labels, include_test=False, include_training_dates=True
    )
    order_rows = labels[ORDER_KEY + ["customer_id"]].reset_index(drop=True).merge(
        orders[ORDER_KEY + ["order_date", "order_value"]],
        on=ORDER_KEY, how="left", validate="one_to_one", sort=False,
    )
    order_rows = order_rows.merge(
        customers[CUSTOMER_KEY + ["first_order_datetime"]],
        on=CUSTOMER_KEY, how="left", validate="many_to_one", sort=False,
    )
    order_date = pd.to_datetime(order_rows["order_date"]).dt.normalize()
    first_order = pd.to_datetime(order_rows["first_order_datetime"]).dt.normalize()
    history = (order_date - first_order).dt.days.astype(float)
    history = history.where(history >= 0)
    prepared = []
    for frame in (x_train, x_val):
        rows = order_rows.loc[frame.index]
        if not np.allclose(rows["order_value"].astype(float), frame["order_value"], equal_nan=True):
            raise ValueError("Customer history is misaligned with PBI 03 order rows.")
        prepared.append(frame.assign(**{HISTORY: history.loc[frame.index]}))
    report = {key: value for key, value in report.items() if key != "training_dates"}
    return prepared[0], y_train, prepared[1], y_val, report, dates


def fit_model(features, labels, estimator):
    shared = fit_preprocessor(features[INPUT_FEATURES])
    history = ColumnTransformer([
        ("customer_history", SimpleImputer(strategy="median", keep_empty_features=True), [HISTORY]),
    ]).fit(features)
    preprocessor = FeatureUnion([("shared", shared), ("history", history)])
    estimator.fit(preprocessor.transform(features), labels)
    return Pipeline([("preprocessor", preprocessor), ("classifier", estimator)])


def score_model(model, features):
    positive = list(model.named_steps["classifier"].classes_).index(1)
    return model.predict_proba(features)[:, positive]


def evaluate(labels, scores):
    predicted = (scores >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "f1": f1_score(labels, predicted, zero_division=0),
        "average_precision": average_precision_score(labels, scores),
        "roc_auc": roc_auc_score(labels, scores),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "flag_rate": float(predicted.mean()),
    }


def log_verified_run(name, model, metrics, split_report, diagnosis, x_val, tracking_uri):
    import mlflow
    import mlflow.sklearn

    if urlsplit(tracking_uri).scheme not in {"http", "https"}:
        raise ValueError("Use the team's shared HTTP(S) MLflow tracking URI.")
    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment("PBI08_Ephraim")
    with mlflow.start_run(run_name=name, tags={"member": "Ephraim", "pbi": "08", "version": name}) as run:
        mlflow.log_params(model.named_steps["classifier"].get_params())
        mlflow.log_param("threshold", THRESHOLD)
        mlflow.log_metrics(metrics)
        mlflow.log_dict(split_report, "split.json")
        mlflow.log_dict(diagnosis, "diagnosis.json")
        mlflow.log_dict({
            "features": list(x_val.columns),
            "preprocessing": {
                "amounts": "median imputation, log1p, robust scaling",
                "order_dates": "median imputation",
                "categories": "missing indicator category and one-hot encoding",
                "customer_history": "median imputation",
            },
            "dependencies": {p: version(p) for p in ("numpy", "pandas", "scikit-learn", "mlflow")},
        }, "model_inputs.json")
        model_info = mlflow.sklearn.log_model(model, artifact_path="model")
        sample = x_val.head(10)
        restored = mlflow.sklearn.load_model(model_info.model_uri)
        np.testing.assert_allclose(score_model(restored, sample), score_model(model, sample))
        mlflow.log_metric("artifact_reload_verified", 1)
    run_id = run.info.run_id
    return {
        "run_id": run_id, "model_uri": model_info.model_uri,
        "run_url": f"{tracking_uri.rstrip('/')}/#/experiments/{run.info.experiment_id}/runs/{run_id}",
        "artifact_reload_verified": True,
    }


def train_ephraim_models(x_train, y_train, x_val, y_val, split_report, train_dates, tracking_uri=None):
    """Train two decision trees and optionally verify artifacts on team MLflow."""
    if not train_dates.index.equals(x_train.index):
        raise ValueError("Training dates must align with the training frame.")
    if split_report["train_last_date"] >= split_report["validation_first_date"]:
        raise ValueError("Training dates must precede validation.")
    if len(x_train) != split_report["training_rows"] or len(x_val) != split_report["validation_rows"]:
        raise ValueError("Prepared row counts must match the split report.")

    unique_dates = pd.Index(train_dates.unique()).sort_values()
    if len(unique_dates) < 2:
        raise ValueError("Use at least two distinct training dates for the diagnostic.")
    cut = unique_dates[min(int(len(unique_dates) * 0.80), len(unique_dates) - 1)]
    older = train_dates < cut
    fit_x, fit_y = x_train.loc[older], y_train.loc[older]
    check_x, check_y = x_train.loc[~older], y_train.loc[~older]
    for target in (fit_y, check_y, y_val):
        if set(target) != {0, 1}:
            raise ValueError("Fit, diagnostic and validation labels must each contain fraud and legitimate orders.")

    trial = fit_model(fit_x, fit_y, DecisionTreeClassifier(random_state=42))
    fit_metrics = evaluate(fit_y, score_model(trial, fit_x))
    check_metrics = evaluate(check_y, score_model(trial, check_x))
    gap = fit_metrics["f1"] - check_metrics["f1"]
    if gap >= 0.10:
        change = {"max_depth": 8}
        reason = "The early-fit F1 exceeds the later-training F1 by at least 0.10. Limit depth to reduce variance."
    elif check_metrics["fn"] > check_metrics["fp"]:
        change = {"class_weight": "balanced"}
        reason = "Missed fraud outnumbers false alarms in the diagnostic slice. Weight the minority class."
    else:
        change = {"min_samples_leaf": 20}
        reason = "False alarms meet or exceed missed fraud. Test larger leaves to reduce noisy splits."
    diagnosis = {
        "fit_rows": len(fit_x), "diagnostic_rows": len(check_x),
        "fit_first_date": str(train_dates.loc[older].min().date()),
        "fit_last_date": str(train_dates.loc[older].max().date()),
        "diagnostic_first_date": str(cut.date()),
        "diagnostic_last_date": str(train_dates.max().date()),
        "fit_f1": float(fit_metrics["f1"]), "diagnostic_f1": float(check_metrics["f1"]),
        "diagnostic_fn": check_metrics["fn"], "diagnostic_fp": check_metrics["fp"],
        "change": change, "reason": reason,
    }

    dummy = fit_model(x_train, y_train, DummyClassifier(strategy="most_frequent"))
    # Match the original dummy classifier's constant predictions.
    dummy_predictions = dummy.predict(x_val)
    results = {"dummy": evaluate(y_val, dummy_predictions), "diagnosis": diagnosis, "versions": {}}
    tracking_uri = tracking_uri or os.getenv("MLFLOW_TRACKING_URI")
    for name, parameters in (("v1", {}), ("v2", change)):
        model = fit_model(x_train, y_train, DecisionTreeClassifier(random_state=42, **parameters))
        metrics = evaluate(y_val, score_model(model, x_val))
        evidence = log_verified_run(name, model, metrics, split_report, diagnosis, x_val, tracking_uri) if tracking_uri else {}
        results["versions"][name] = {
            "model": model, "metrics": metrics, "threshold": THRESHOLD, **evidence,
        }
    return results

## 2. Load and check the shared base data
Use PBI 02 cleaning and PBI 03’s chronological 70/15/15 split, with entire dates kept together. The final period stays reserved for PBI 09. Capture data hashes, package versions and split dates/counts for reproduction. Customer history uses first recorded purchase date, so it can be unknown for unmatched customers or orders earlier than the recorded date.

In [ ]:
import hashlib
import subprocess
from pathlib import Path
from importlib.metadata import version

import pandas as pd
from src.data_cleaning import clean_customers, clean_orders, clean_labels

paths = [Path(f"{name}_base.csv") for name in ("customers", "orders", "labels")]
missing = [str(path) for path in paths if not path.is_file()]
if missing:
    raise FileNotFoundError("Place the authorised base files in the repo root: " + ", ".join(missing))

customers, orders, labels = [
    clean(pd.read_csv(path, low_memory=False))[0]
    for path, clean in zip(paths, (clean_customers, clean_orders, clean_labels))
]
x_train, y_train, x_val, y_val, split_report, train_dates = prepare_ephraim_data(customers, orders, labels)
split_report["csv_sha256"] = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in paths}
split_report["git_sha"] = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
split_report["training_fraud_rate"] = float(y_train.mean())
split_report["validation_fraud_rate"] = float(y_val.mean())
display(pd.Series(split_report))
print({package: version(package) for package in ("numpy", "pandas", "scikit-learn", "matplotlib", "seaborn", "mlflow")})

## 3. Train and explain both versions
V1 is the seeded AIML decision tree. A diagnostic copy fits the older training dates and scores the newest 20% of unique training dates. The diagnostic selects one candidate change before validation is used. A 0.10 F1 gap triggers a depth cap. Otherwise the diagnostic errors guide a class-weight or minimum-leaf-size change. These are experiment rules. The validation metrics show whether the chosen change actually helps.

Refit both versions on the same full training period, then score the shared validation period at a fixed 0.50 threshold. If the shared URI is configured, log each full pipeline and reload it for a prediction check. Run without the URI for a local rehearsal.

In [ ]:
import os
from pprint import pprint

results = train_ephraim_models(x_train, y_train, x_val, y_val, split_report, train_dates)
pprint(results["diagnosis"])
if not os.getenv("MLFLOW_TRACKING_URI"):
    print("Local rehearsal. Set MLFLOW_TRACKING_URI to collect shared run and reload evidence.")

## 4. Compare validation results
Precision measures the share of flagged orders that were fraud. Recall measures the share of actual fraud caught. F1 combines both. Average precision and ROC-AUC measure ranking. Compare false alarms, missed fraud and the fraction sent for review. Record the measured outcome even if v2 performs worse.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

comparison = pd.DataFrame({
    "dummy": results["dummy"],
    **{name: result["metrics"] for name, result in results["versions"].items()},
})
comparison["v2 minus v1"] = comparison["v2"] - comparison["v1"]
display(comparison.round(4))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, (name, result) in zip(axes, results["versions"].items()):
    metrics = result["metrics"]
    cm = [[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Legitimate", "Fraud"], yticklabels=["Legitimate", "Fraud"])
    ax.set(xlabel="Predicted", ylabel="Actual", title=name)
    predictions = (score_model(result["model"], x_val) >= result["threshold"]).astype(int)
    print(name, classification_report(y_val, predictions, labels=[0, 1], zero_division=0))
fig.tight_layout()
plt.show()

delta = comparison["v2 minus v1"]
print("Measured outcome at 0.50")
print(f"F1 {delta['f1']:+.4f}, recall {delta['recall']:+.4f}, "
      f"false alarms {delta['fp']:+.0f}, missed fraud {delta['fn']:+.0f}.")

## 5. Retain shared MLflow and PBI 10 evidence
For each version, retain its run URL, model URI and successful reload output. In a separate Python process, set the same tracking URI and load the URI with `mlflow.sklearn.load_model` to confirm another client can retrieve it. Apply the recorded threshold to the fraud probability.

For a headless demonstration of the callable, run the notebook from the repository root:

```bash
jupyter nbconvert --to notebook --execute --stdout KochuveettilTojiEphraimModels.ipynb
```

The notebook calls `prepare_ephraim_data` and `train_ephraim_models` directly. The headless command demonstrates the same calls on the base CSVs. The batch workflow still needs to pass its cleaned batch inputs and demonstrate integration for PBI 10.

In [ ]:
for name, result in results["versions"].items():
    if result.get("artifact_reload_verified"):
        print(name, result["run_url"], result["model_uri"], "reload verified")
    else:
        print(name, "shared run and artifact reload pending")

## Submission checks
- [ ] Run both fitted models on the authorised base data and keep the validation table.
- [ ] Record the reproducible split, source SHA, CSV hashes and evaluation rationale.
- [ ] Retain evidence of the v1 diagnosis, the change tested and the measured v2 outcome.
- [ ] Attach both shared run URLs and confirm model retrieval from a separate client.
- [ ] Demonstrate the PBI 10 training call and keep its invocation/output.
- [ ] Obtain and retain a teammate review and link the evidence in PBI 08.